In [ ]:

from sklearn.decomposition import PCA
from scipy.io import loadmat
from scipy.sparse import csr_array
from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import cdist
from math import sqrt
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.cbook import get_sample_data
from matplotlib.offsetbox import (AnnotationBbox, DrawingArea, OffsetImage,
                                  TextArea)
import numpy as np

# -----------------------------------------------------------------------------
# NOTE: Do not change the parameters / return types for pre defined methods.
# -----------------------------------------------------------------------------
class OrderOfFaces:
    """
    This class handles loading and processing facial image data for dimensionality
    reduction using the ISOMAP algorithm, with PCA as an optional comparison.

    Attributes:
    ----------
    images_path : str
        Path to the .mat file containing the image dataset.

    Methods:
    -------
    get_adjacency_matrix(epsilon):
        Returns the adjacency matrix based on a given epsilon neighborhood.

    get_best_epsilon():
        Returns the best epsilon for the ISOMAP algorithm, likely based on
        graph connectivity or reconstruction error.

    isomap(epsilon):
        Computes a 2D embedding of the data using the ISOMAP algorithm.

    pca(num_dim):
        Returns a low-dimensional embedding of the data using PCA.
    """

    def __init__(self, images_path="data/isomap.mat"):
        """
        Initializes the OrderOfFaces object and loads image data from the given path.

        Parameters:
        ----------
        images_path : str
            Path to the .mat file containing the facial images dataset.
        """

        self.datas = loadmat(
            images_path
        )  # https://stackoverflow.com/questions/874461/read-mat-files-in-python
        data_array = np.array(self.datas["images"])
        data_array_t = data_array.T
        self.data_array = data_array

        self.data_array_t = data_array_t
        data_norm = np.linalg.norm(data_array_t, axis=1, keepdims=True)
        data_normalized = data_array_t / data_norm
        self.data_norm = data_normalized
        self.adjancy_matrix_viz = None

    def get_adjacency_matrix(self, epsilon: float) -> np.ndarray:
        """
        Constructs the adjacency matrix using epsilon neighborhoods.

        Parameters:
        ----------
        epsilon : float
            The neighborhood radius within which points are considered connected.

        Returns:
        -------
        np.ndarray
            A 2D adjacency matrix (m x m) where each entry represents distance between
            neighbors within the epsilon threshold.
        """

        distance_matrix = cdist(
            self.data_array_t, self.data_array_t, metric="euclidean"
        ).astype(
            np.float64
        )  # https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html

        adjancy_matrix = np.where(
            distance_matrix <= epsilon, distance_matrix, 0
        ).astype(np.float64)
        np.fill_diagonal(adjancy_matrix, 0.0)

        adjancy_matrix_viz = np.where(
            distance_matrix <= epsilon, 1, 0
        ).astype(np.float64)
        np.fill_diagonal(adjancy_matrix, 0.0)

        self.adjancy_matrix = adjancy_matrix
        self.adjancy_matrix_viz = adjancy_matrix_viz

        return adjancy_matrix

    def get_best_epsilon(self) -> float:
        """
        Heuristically determines the best epsilon value for graph connectivity in ISOMAP.

        Returns:
        -------
        float
            Optimal epsilon value ensuring a well-connected neighborhood graph.


        """

        return 14.63
        raise NotImplementedError("Not Implemented")

    def isomap(self, epsilon: float) -> np.ndarray:
        """
        Applies the ISOMAP algorithm to compute a 2D low-dimensional embedding of the dataset.

        Parameters:
        ----------
        epsilon : float
            The neighborhood radius for building the adjacency graph.

        Returns:
        -------
        np.ndarray
            A (m x 2) array where each row is a 2D embedding of the original data point.


        """

        adj_matrix = self.get_adjacency_matrix(epsilon)

        graph = csr_array(adj_matrix)
        short_path_matrix = shortest_path(
            csgraph=graph, directed=False, method="FW"
        )  # https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csgraph.shortest_path.html
        I = np.identity(short_path_matrix.shape[0])

        ones = np.ones((short_path_matrix.shape[0], 1))

        H = I - (1 / short_path_matrix.shape[0]) * (ones @ ones.T)

        G = (-1 / 2) * H @ (short_path_matrix**2) @ H

        e_vals, e_vectors = np.linalg.eigh(G)
        idx = np.argsort(e_vals)[
            ::-1
        ]  # https://stackoverflow.com/questions/8092920/sort-eigenvalues-and-associated-eigenvectors-after-using-numpy-linalg-eig-in-pyt
        e_vals = e_vals[idx]
        e_vectors = e_vectors[:, idx]

        Z = e_vectors[:, :2] * np.sqrt(e_vals[:2])

        return Z

    def pca(self, num_dim: int) -> np.ndarray:
        """
        Applies PCA to reduce the dataset to a specified number of dimensions.

        Parameters:
        ----------
        num_dim : int
            Number of principal components to project the data onto.

        Returns:
        -------
        np.ndarray
            A (m x num_dim) array representing the dataset in a reduced PCA space.


        """
        pca = PCA(n_components=num_dim)

        pca_embedding = pca.fit_transform(self.data_array_t)
        return pca_embedding


In [ ]:
def creat_graph_w_images_scatter_pca(display_nodes, raw_image_data, Z,ns):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(Z[:, 0], Z[:, 1], s=6)
    # ax.invert_yaxis()



    display_locations = []
    for i,n in enumerate((display_nodes)):
        xy = (Z[n, 0], Z[n, 1])
        top_move = 0
        bottom_move =0
        if xy[0] <0 :
            display_locations.append((0,-185+bottom_move)) 
            bottom_move+=10
        else:
            display_locations.append((0,185+top_move)) 
            top_move+=10  

    for node, loc in zip(display_nodes, display_locations):
        xy = (Z[node, 0], Z[node, 1])

        arr_img = raw_image_data[:, node].reshape(64, 64).T

        ax.plot(xy[0], xy[1], ".r")

        offset = np.array([0.02, 0.02])

        imagebox = OffsetImage(arr_img, zoom=0.4)
        ab = AnnotationBbox( #https://matplotlib.org/stable/gallery/text_labels_and_annotations/demo_annotation_box.html)
            imagebox,
            xy,
            xybox=loc,
            xycoords='data',
            boxcoords="offset points",
            pad=.3,
            arrowprops=dict(
                arrowstyle="->",color=
            'red'
            )
        )

        ax.add_artist(ab)

    plt.title(f'PCA: {ns} Components')

    plt.show()


In [ ]:
def creat_graph_w_images(display_nodes:list, raw_image_data, adj_matrix):
    fig, ax = plt.subplots()

    rows, cols = np.where(np.triu(adj_matrix, k=1) == 1) 
    edges = list(zip(rows.tolist(), cols.tolist()))

    gr = nx.Graph() #https://www.geeksforgeeks.org/python/networkx-python-software-package-study-complex-networks/
    gr.add_edges_from(edges)
    gr.add_nodes_from(range(adj_matrix.shape[0]))

    pos = nx.spring_layout(gr, seed=42)

    nx.draw(gr, pos,
            with_labels=False,
            node_color='blue',
            node_size=20,
            edge_color='gray',
            width=.1,
            ax=ax)

   
    display_locations = []
    for i in range(len(display_nodes)):
        top_move = 0
        bottom_move =0
        if i < 4:
            display_locations.append((0,185+top_move)) 
            top_move+=10  
        else:
            display_locations.append((0,-185+bottom_move)) 
            bottom_move+=10

    for node, loc in zip(display_nodes, display_locations):
        
        arr_img = raw_image_data[:,node].reshape(64,64).T
        xy = pos[node]
        ax.plot(xy[0], xy[1], ".r")

        imagebox = OffsetImage(arr_img, zoom=.6)
        imagebox.image.axes = ax

        ab = AnnotationBbox( #https://matplotlib.org/stable/gallery/text_labels_and_annotations/demo_annotation_box.html)
            imagebox,
            xy,
            xybox=loc,
            xycoords='data',
            boxcoords="offset points",
            pad=.3,
            arrowprops=dict(
                arrowstyle="->",color=
            'red'
            )
        )

        ax.add_artist(ab)

    plt.show()


In [ ]:
def creat_graph_w_images_scatter(display_nodes, raw_image_data, Z,e):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.scatter(Z[:, 0], Z[:, 1], s=6)
    # ax.invert_yaxis()



    display_locations = []
    for i,n in enumerate((display_nodes)):
        xy = (Z[n, 0], Z[n, 1])
        top_move = 0
        bottom_move =0
        if xy[0] <0 :
            display_locations.append((0,-185+bottom_move)) 
            bottom_move+=10
        else:
            display_locations.append((0,185+top_move)) 
            top_move+=10  

    for node, loc in zip(display_nodes, display_locations):
        xy = (Z[node, 0], Z[node, 1])

        arr_img = raw_image_data[:, node].reshape(64, 64).T

        ax.plot(xy[0], xy[1], ".r")

        offset = np.array([0.02, 0.02])

        imagebox = OffsetImage(arr_img, zoom=0.4)
        ab = AnnotationBbox( #https://matplotlib.org/stable/gallery/text_labels_and_annotations/demo_annotation_box.html)
            imagebox,
            xy,
            xybox=loc,
            xycoords='data',
            boxcoords="offset points",
            pad=.3,
            arrowprops=dict(
                arrowstyle="->",color=
            'red'
            )
        )

        ax.add_artist(ab)

    plt.title(f'Epsilon: {e}')

    plt.show()


In [ ]:
oof = OrderOfFaces()

In [ ]:
addj_matrix = oof.get_adjacency_matrix(11.5)
Z=oof.isomap(11.5)

Q3.1

In [ ]:
display_nodes = [10, 0, 581,333,423,215,5,53]
creat_graph_w_images(display_nodes,oof.data_array,oof.adjancy_matrix_viz)

Q3.2

- The embeddings as a result of the Epsilon- ISOMAP show a clear pattern of faces being organized based on their pose orientation. Based on the vectorized represntation of each image, images with similar orietation have smaller pairwise Euclidean distance between them and are determined to be within the epsilon threshold. Once images have determind to be in one another neighborhood, finding the geodesic distance via shortest path distance between images allows us to create a nonlinear manifold representation of the data. The resulting low dimensional embbedings show a gradual shift in face orientation as we move throughout the graph.
- Based on the sample images plotted, we can see evidence of similar face orientations being grouped closer together. Additionally, when moving around the graph we see evidence of the gradual shift in orientation of the faces displayed.  
- The results of the Epsilon- ISOMAP implementation appear very similar to the output of the K-ISOMAP algorith used in the paper. The resulting structure of the grouped nodes appear almost identical to eachother. 
- In order to tune the Epsilon value, values between 10.5 and 21 were itterated through and the resulting manifold was plotted. Smaller values of Epsilon less than 10.5 tended to result in fragmented graphs, not correctly preserving the the relationships between picture orientation. Larger values of Epsilon greater than 16 tended to incorrectly associate images with more dissimilarties thus reducing the difference between facial orientation. Based on these test and comparing the output to the graph in the paper, the ideal value of Epsilon was 11.375 as the transition between images showed gradual changes in orientation. 


In [ ]:

    
for e in np.linspace(10.5, 21 ,25):
    addj_matrix = oof.get_adjacency_matrix(e)
    Z=oof.isomap(e)
    print(e)
    nodes = [
    np.argmin(Z[:,0]),
    np.argmax(Z[:,0]),
    np.argmin(Z[:,1]),
    np.argmax(Z[:,1]),
    np.argmin(np.linalg.norm(Z, axis=1))
]
    
    n_random = 4

    all_indices = np.arange(Z.shape[0])
    remaining = np.setdiff1d(all_indices, nodes)

    random_nodes = np.random.choice(
        remaining,
        size=min(n_random, len(remaining)),
        replace=False
    )

    nodes.extend(random_nodes.tolist())
    
    
    if len(Z)==0:
        pass
    else:
        
        creat_graph_w_images_scatter(nodes,oof.data_array,Z,e)
    # plt.scatter(Z[:,0], Z[:,1])
# plt.show()


Q3.3

- Based on the visual output from PCA on the face images, there seems to be a more meaningful projection when using Epsilon- ISOMAP. I believe this is due to the method in which PCA decostructs data. When using only the 2 components, PCA determines the two principal components the result in the most variance between datapoints in a linear projection. Therefore, depending on which two components are selected, more gradual transitions between images orientation can be missed when using PCA if the components selected focus on variance based on lighting. 


-  Based on the visual output from PCA on the face images, there seems to be a more meaningful projection when using Epsilon- ISOMAP. PCA identifies the principal components that captures the largerst variance between datapoints through linear combinations. Due to PCA operating in a linear setting, it may exclude the nonlinear factors the result in identification of gradual orientation change. If the principal components identify factors such as lighting to be the main cause of variance, the picture groupings may not display a smooth transition of facial orientation like ISOMAP.  

In [ ]:
pca_viz = oof.pca(2)

In [ ]:
creat_graph_w_images_scatter_pca(nodes,oof.data_array,pca_viz,2)